In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

# --- PATH SETUP (Repository-Aware) ---
# Detects the current folder (e.g., .../data/notebooks)
current_dir = os.getcwd() 

# Use '..' to go up to the main 'data' folder, then into 'raw' or 'processed'
raw_data_path = os.path.normpath(os.path.join(current_dir, '..', 'raw'))
processed_data_path = os.path.normpath(os.path.join(current_dir, '..', 'processed'))

# Ensure the processed directory exists
os.makedirs(processed_data_path, exist_ok=True)

# --- 1. NASA & NOAA (Weather) ---
print("Processing Weather...")
nasa_df = pd.read_csv(os.path.join(raw_data_path, 'NASA Power DAV.csv'), skiprows=15)
nasa_df['date'] = nasa_df.apply(lambda row: datetime(int(row['YEAR']), 1, 1) + timedelta(days=int(row['DOY']) - 1), axis=1)
nasa_df['month'] = nasa_df['date'].dt.month
nasa_df['year'] = nasa_df['date'].dt.year.astype(int) 
nasa_df = nasa_df[['date', 'year', 'month', 'PRECTOTCORR', 'T2M']].replace(-999, np.nan)

noaa_df = pd.read_csv(os.path.join(raw_data_path, 'NOAA.csv'), low_memory=False)
noaa_df['date'] = pd.to_datetime(noaa_df['DATE'])
noaa_daily = noaa_df.groupby('date')[['PRCP', 'TAVG']].mean().reset_index()

weather_daily = pd.merge(nasa_df, noaa_daily, on='date', how='outer')

# --- 2. TRADE DATA ---
print("Processing Trade...")
trade_files = ['TradeData_4_21_2026_23_16_47.csv', 'TradeData_4_21_2026_23_14_27.csv', 'TradeData_4_21_2026_23_11_59.csv']
# We look for these specifically in data/raw
trade_list = [pd.read_csv(os.path.join(raw_data_path, f)) for f in trade_files]
trade_df = pd.concat(trade_list)

trade_yearly = trade_df.groupby('refYear')[['primaryValue', 'qty']].sum().reset_index()
trade_yearly['year'] = trade_yearly['refYear'].astype(int) 
trade_yearly = trade_yearly.drop(columns=['refYear'])

# --- 3. WFP DATA ---
print("Processing WFP Philippines...")
wfp_df = pd.read_csv(os.path.join(raw_data_path, 'globalfoodprices_wfp.csv'), low_memory=False)
wfp_df = wfp_df[wfp_df['adm0_name'] == 'Philippines'].copy()
wfp_df['year'] = wfp_df['mp_year'].astype(int) 
wfp_df['month'] = wfp_df['mp_month'].astype(int)
wfp_df = wfp_df[['adm1_name', 'mkt_name', 'cm_name', 'year', 'month', 'mp_price']]

# --- 4. MASTER MERGE ---
print("Merging 2.5 Million Rows...")
master_df = pd.merge(wfp_df, weather_daily, on=['year', 'month'], how='left')
final_df = pd.merge(master_df, trade_yearly, on='year', how='left')

# --- 5. CLEANUP & EXPORT ---
final_df = final_df.dropna(subset=['mp_price', 'date'])

# Downcast memory usage
fcols = final_df.select_dtypes('float').columns
final_df[fcols] = final_df[fcols].apply(pd.to_numeric, downcast='float')

print(f"\n✅ SUCCESS! Total Rows: {final_df.shape[0]:,}")

# Export to data/processed
output_filename = 'final_master_dataset_fixed.csv'
output_path = os.path.join(processed_data_path, output_filename)
final_df.to_csv(output_path, index=False)

print(f"Final file saved to: {output_path}")